In [3]:
#import
import numpy as np
import pandas as pd
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [4]:
#load csv
clinical_df = pd.read_csv("clinical_data.csv")
print(clinical_df.head())

   age_at_prostatectomy  primary_gleason  secondary_gleason  tertiary_gleason  \
0                    66                3                  4               5.0   
1                    60                3                  2               4.0   
2                    66                4                  3               NaN   
3                    69                3                  4               NaN   
4                    65                3                  4               NaN   

   ISUP  pre_operative_PSA  BCR BCR_PSA  time_to_follow-up/BCR pT_stage  \
0     2                8.3  1.0    0.11                    1.3       3b   
1     1               10.0  1.0     1.6                    1.7       3a   
2     3                6.9  0.0     NaN                   60.7       2c   
3     2               15.0  0.0     NaN                   60.6       3a   
4     2                5.2  0.0     NaN                   62.3       2c   

  positive_lymph_nodes capsular_penetration  positive_surgical

In [5]:
#train and test split
TRAIN_BCR_POS = ['1003', '1010', '1030', '1041', '1064', '1086', '1100', '1106',
                 '1127', '1135', '1137', '1165', '1169', '1185', '1195', '1208',
                 '1214', '1217', '1219', '1240', '1258', '1287']
TRAIN_BCR_NEG = ['1021', '1026', '1028', '1031', '1035', '1048', '1052', '1056',
                 '1060', '1062', '1066', '1071', '1094', '1112', '1114', '1117',
                 '1123', '1129', '1136', '1140', '1141', '1149', '1174', '1179',
                 '1186', '1188', '1192', '1205', '1206', '1207', '1211', '1212',
                 '1216', '1223', '1227', '1260', '1261', '1262', '1264', '1269',
                 '1279', '1282', '1284', '1285', '1290', '1291', '1293', '1294',
                 '1296', '1298', '1299', '1301', '1303', '1304']
TEST_BCR_POS  = ['1090', '1122', '1130', '1193', '1295']
TEST_BCR_NEG  = ['1011', '1025', '1036', '1037', '1039', '1068', '1107', '1120',
                 '1151', '1184', '1235', '1252', '1267', '1286']

all_train_ids = [int(x) for x in TRAIN_BCR_POS + TRAIN_BCR_NEG]
all_test_ids  = [int(x) for x in TEST_BCR_POS  + TEST_BCR_NEG]

print(f'Train patients: {len(all_train_ids)} ({len(TRAIN_BCR_POS)} BCR+, {len(TRAIN_BCR_NEG)} BCR-)')
print(f'Test  patients: {len(all_test_ids)}  ({len(TEST_BCR_POS)} BCR+, {len(TEST_BCR_NEG)} BCR-)')

Train patients: 76 (22 BCR+, 54 BCR-)
Test  patients: 19  (5 BCR+, 14 BCR-)


In [6]:
#drop columns that are redundant: primary/secondary/tertiary_gleason: ISUP is better, 
#time_to_follow-up/BCR and BCR_PSA: only exist for BCR+ patients
#positive_lymph_nodes: too many missing values
clinical_df = clinical_df.drop(
    ['primary_gleason', 'secondary_gleason', 'tertiary_gleason',
     'time_to_follow-up/BCR', 'BCR_PSA', 'positive_lymph_nodes'],
    axis=1
)
print('Remaining columns:', clinical_df.columns.tolist())

Remaining columns: ['age_at_prostatectomy', 'ISUP', 'pre_operative_PSA', 'BCR', 'pT_stage', 'capsular_penetration', 'positive_surgical_margins', 'invasion_seminal_vesicles', 'lymphovascular_invasion', 'earlier_therapy', 'patient_id']


In [7]:
#capsular_penetration has 'x' for unknown values, replace 'x' with the mode from train patients to keep feature
print('capsular_penetration unique values:', clinical_df['capsular_penetration'].unique())
clinical_df['capsular_penetration'] = clinical_df['capsular_penetration'].replace('x', np.nan)
clinical_df['capsular_penetration'] = pd.to_numeric(clinical_df['capsular_penetration'], errors='coerce')

#compute mode from training patients only so nothing is leaked
train_cap_pen_mode = clinical_df.loc[
    clinical_df['patient_id'].isin(all_train_ids), 'capsular_penetration'
].mode()[0]
print(f'Imputing capsular_penetration missing values with mode: {train_cap_pen_mode}')
clinical_df['capsular_penetration'] = clinical_df['capsular_penetration'].fillna(train_cap_pen_mode)

capsular_penetration unique values: <StringArray>
['1', 'x', '0']
Length: 3, dtype: str
Imputing capsular_penetration missing values with mode: 0.0


In [8]:
#one-hot encodings
clinical_df = pd.get_dummies(clinical_df, columns=['pT_stage', 'earlier_therapy'],
                              drop_first=True, dtype=int)

#fix dtypes
clinical_df['BCR'] = clinical_df['BCR'].astype('int64')
clinical_df['lymphovascular_invasion'] = clinical_df['lymphovascular_invasion'].astype('int64')
clinical_df['capsular_penetration'] = clinical_df['capsular_penetration'].astype('int64')

print('Final columns:', clinical_df.columns.tolist())
print('Shape:', clinical_df.shape)

Final columns: ['age_at_prostatectomy', 'ISUP', 'pre_operative_PSA', 'BCR', 'capsular_penetration', 'positive_surgical_margins', 'invasion_seminal_vesicles', 'lymphovascular_invasion', 'patient_id', 'pT_stage_2a', 'pT_stage_2c', 'pT_stage_3a', 'pT_stage_3b', 'pT_stage_4', 'pT_stage_4b', 'earlier_therapy_radiotherapy + cryotherapy', 'earlier_therapy_radiotherapy + hormones', 'earlier_therapy_unknown']
Shape: (95, 18)


In [9]:
#split into train and test dataframes
train_df = clinical_df[clinical_df['patient_id'].isin(all_train_ids)].reset_index(drop=True)
test_df  = clinical_df[clinical_df['patient_id'].isin(all_test_ids)].reset_index(drop=True)

print(f'Train df: {train_df.shape}, Test df: {test_df.shape}')

#separate features (X) from labels (y)
label_cols = ['BCR', 'patient_id']
feature_cols = [c for c in train_df.columns if c not in label_cols]

X_train = train_df[feature_cols].copy()
y_train = train_df[label_cols].copy()

X_test  = test_df[feature_cols].copy()
y_test  = test_df[label_cols].copy()

print(f'Feature count: {len(feature_cols)}')
print('Features:', feature_cols)

Train df: (76, 18), Test df: (19, 18)
Feature count: 16
Features: ['age_at_prostatectomy', 'ISUP', 'pre_operative_PSA', 'capsular_penetration', 'positive_surgical_margins', 'invasion_seminal_vesicles', 'lymphovascular_invasion', 'pT_stage_2a', 'pT_stage_2c', 'pT_stage_3a', 'pT_stage_3b', 'pT_stage_4', 'pT_stage_4b', 'earlier_therapy_radiotherapy + cryotherapy', 'earlier_therapy_radiotherapy + hormones', 'earlier_therapy_unknown']


In [10]:
#log-transform PSA (right-skewed)
X_train['pre_operative_PSA'] = np.log1p(X_train['pre_operative_PSA'])
X_test['pre_operative_PSA']  = np.log1p(X_test['pre_operative_PSA'])

continuous_cols = ['age_at_prostatectomy', 'ISUP', 'pre_operative_PSA']
binary_cols = [col for col in feature_cols if col not in continuous_cols]

In [11]:
EMBED_DIM = 64  #to match WSI and MRI embeddings

class DenoisingAutoencoder(nn.Module):
    def __init__(self, input_dim, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        return out, z

In [12]:
#stratified split preserves BCR+ / BCR- ratio in each fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

embedding_rows = []  #for train fold embeddings (cross-val)

X_train_array = X_train.values.astype(np.float32)
y_labels = y_train['BCR'].values  #for stratification

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_array, y_labels), start=1):
    print(f'\nFold {fold}')

    X_tr = X_train.iloc[tr_idx].copy()
    X_val = X_train.iloc[val_idx].copy()

    #fit scaler only on training fold
    scaler = StandardScaler()
    X_tr[continuous_cols]  = scaler.fit_transform(X_tr[continuous_cols])
    X_val[continuous_cols] = scaler.transform(X_val[continuous_cols])

    X_tr_arr  = X_tr.values.astype(np.float32)
    X_val_arr = X_val.values.astype(np.float32)

    train_loader = DataLoader(TensorDataset(torch.tensor(X_tr_arr)),
                              batch_size=16, shuffle=True)
    val_loader   = DataLoader(TensorDataset(torch.tensor(X_val_arr)),
                              batch_size=16, shuffle=False)

    input_dim = X_tr_arr.shape[1]
    model = DenoisingAutoencoder(input_dim, embed_dim=EMBED_DIM).to(device)

    cont_idx = [X_tr.columns.get_loc(c) for c in continuous_cols]
    bin_idx  = [X_tr.columns.get_loc(c) for c in binary_cols]

    mse_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    def compute_loss(recon, target):
        loss_cont = mse_loss(recon[:, cont_idx], target[:, cont_idx])
        loss_bin  = bce_loss(recon[:, bin_idx],  target[:, bin_idx])
        return loss_cont + loss_bin, loss_cont, loss_bin

    best_val_loss = float('inf')
    best_state = None
    patience, patience_counter = 20, 0
    noise_std = 0.1

    for epoch in range(300):
        model.train()
        train_loss = 0.0
        for (batch,) in train_loader:
            batch = batch.to(device)
            noisy = batch + noise_std * torch.randn_like(batch)
            optimizer.zero_grad()
            recon, _ = model(noisy)
            loss, _, _ = compute_loss(recon, batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(device)
                recon, _ = model(batch)
                loss, _, _ = compute_loss(recon, batch)
                val_loss += loss.item()
        val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 50 == 0:
            print(f'  Epoch {epoch}: train={train_loss:.4f}, val={val_loss:.4f}')
        if patience_counter >= patience:
            print(f'  Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    model.eval()

    X_val_tensor = torch.tensor(X_val_arr).to(device)
    with torch.no_grad():
        _, val_embeddings = model(X_val_tensor)
    val_embeddings = val_embeddings.cpu().numpy()

    for i, row_idx in enumerate(val_idx):
        row = {'patient_id': y_train.iloc[row_idx]['patient_id'],
               'BCR': y_train.iloc[row_idx]['BCR'],
               'fold': fold}
        for j in range(EMBED_DIM):
            row[f'embed_{j+1}'] = val_embeddings[i, j]
        embedding_rows.append(row)

train_embeddings_cv_df = pd.DataFrame(embedding_rows).sort_values('patient_id').reset_index(drop=True)
print(f'\nCross-val embeddings shape: {train_embeddings_cv_df.shape}')
print(train_embeddings_cv_df.head())

Using device: cpu

Fold 1
  Epoch 0: train=1.6109, val=1.6336
  Epoch 50: train=0.1274, val=0.2631
  Epoch 100: train=0.0455, val=0.1632
  Epoch 150: train=0.0231, val=0.1528
  Early stopping at epoch 164

Fold 2
  Epoch 0: train=1.6635, val=1.6632
  Epoch 50: train=0.1455, val=0.2721
  Epoch 100: train=0.0446, val=0.2282
  Early stopping at epoch 121

Fold 3
  Epoch 0: train=1.6455, val=1.3858
  Epoch 50: train=0.1335, val=0.1536
  Epoch 100: train=0.0414, val=0.1503
  Early stopping at epoch 104

Fold 4
  Epoch 0: train=1.6785, val=1.6633
  Epoch 50: train=0.1539, val=0.1959
  Epoch 100: train=0.0479, val=0.0843
  Epoch 150: train=0.0261, val=0.0563
  Early stopping at epoch 185

Fold 5
  Epoch 0: train=1.6470, val=1.9395
  Epoch 50: train=0.1267, val=0.1740
  Epoch 100: train=0.0428, val=0.0837
  Epoch 150: train=0.0239, val=0.0719
  Early stopping at epoch 174

Cross-val embeddings shape: (76, 67)
   patient_id  BCR  fold   embed_1   embed_2   embed_3   embed_4   embed_5  \
0      

In [13]:
#fit final scaler on all training data
final_scaler = StandardScaler()
X_train_final = X_train.copy()
X_test_final  = X_test.copy()

X_train_final[continuous_cols] = final_scaler.fit_transform(X_train_final[continuous_cols])
X_test_final[continuous_cols]  = final_scaler.transform(X_test_final[continuous_cols])

#save scalar
with open('clinical_scaler.pkl', 'wb') as f:
    pickle.dump(final_scaler, f)
print('Scaler saved to clinical_scaler.pkl')

X_train_final_arr = X_train_final.values.astype(np.float32)
X_test_final_arr  = X_test_final.values.astype(np.float32)

full_loader = DataLoader(TensorDataset(torch.tensor(X_train_final_arr)),
                         batch_size=16, shuffle=True)

input_dim = X_train_final_arr.shape[1]
final_model = DenoisingAutoencoder(input_dim, embed_dim=EMBED_DIM).to(device)

cont_idx = [X_train_final.columns.get_loc(c) for c in continuous_cols]
bin_idx  = [X_train_final.columns.get_loc(c) for c in binary_cols]

mse_loss  = nn.MSELoss()
bce_loss  = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=1e-3)

def compute_loss(recon, target):
    loss_cont = mse_loss(recon[:, cont_idx], target[:, cont_idx])
    loss_bin  = bce_loss(recon[:, bin_idx],  target[:, bin_idx])
    return loss_cont + loss_bin

print('Training final model on all training data...')
for epoch in range(300):
    final_model.train()
    epoch_loss = 0.0
    for (batch,) in full_loader:
        batch = batch.to(device)
        noisy = batch + 0.1 * torch.randn_like(batch)
        optimizer.zero_grad()
        recon, _ = final_model(noisy)
        loss = compute_loss(recon, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    if epoch % 50 == 0:
        print(f'  Epoch {epoch}: loss={epoch_loss/len(full_loader):.4f}')

print('Done.')

Scaler saved to clinical_scaler.pkl
Training final model on all training data...
  Epoch 0: loss=1.6355
  Epoch 50: loss=0.1086
  Epoch 100: loss=0.0296
  Epoch 150: loss=0.0166
  Epoch 200: loss=0.0167
  Epoch 250: loss=0.0164
Done.


In [14]:
final_model.eval()

#extract train embeddings
with torch.no_grad():
    _, train_emb = final_model(torch.tensor(X_train_final_arr).to(device))
    _, test_emb  = final_model(torch.tensor(X_test_final_arr).to(device))

train_emb = train_emb.cpu().numpy()
test_emb  = test_emb.cpu().numpy()

#output dataframes
embed_cols = [f'embed_{j+1}' for j in range(EMBED_DIM)]

train_out = pd.DataFrame(train_emb, columns=embed_cols)
train_out.insert(0, 'patient_id', y_train['patient_id'].values)
train_out.insert(1, 'BCR',        y_train['BCR'].values)

test_out = pd.DataFrame(test_emb, columns=embed_cols)
test_out.insert(0, 'patient_id', y_test['patient_id'].values)
test_out.insert(1, 'BCR',        y_test['BCR'].values)

train_out = train_out.sort_values('patient_id').reset_index(drop=True)
test_out  = test_out.sort_values('patient_id').reset_index(drop=True)

print('Train embeddings shape:', train_out.shape)
print('Test embeddings shape: ', test_out.shape)
print(train_out.head())

Train embeddings shape: (76, 66)
Test embeddings shape:  (19, 66)
   patient_id  BCR   embed_1   embed_2   embed_3   embed_4   embed_5  \
0        1003    1 -1.041065 -0.050969 -0.617581 -1.402086 -1.673047   
1        1010    1 -1.395280 -0.064439  0.012714 -1.166207  0.767700   
2        1021    0 -0.529108 -0.206478 -0.298364 -0.481965 -0.368471   
3        1026    0 -0.595712 -0.794656  0.028039 -0.767361 -0.652597   
4        1028    0 -1.187598 -0.887505 -0.139512 -1.254092 -0.770568   

    embed_6   embed_7   embed_8  ...  embed_55  embed_56  embed_57  embed_58  \
0 -1.594541 -1.099066  0.549613  ... -1.726105  0.963005  0.759237 -0.319183   
1 -0.741927 -0.125473  0.668288  ...  0.111575  0.364764  0.625150 -0.369312   
2 -1.173663  1.024253 -0.273682  ...  0.381847 -0.094700  0.864707 -1.625020   
3  0.207885  0.812380  1.028930  ...  0.702819 -0.213325 -0.167091  0.213635   
4  0.008095  0.241999  0.455292  ...  0.305528 -0.382198  0.561116  0.939327   

   embed_59  embed_6

In [15]:
train_out.to_csv('full_clinical_embeddings_train.csv', index=False)
test_out.to_csv('full_clinical_embeddings_test.csv',   index=False)

print(f'\nEmbedding dimensionality: {EMBED_DIM}D')
print('Each CSV has columns: patient_id, BCR, embed_1 ... embed_64')


Embedding dimensionality: 64D
Each CSV has columns: patient_id, BCR, embed_1 ... embed_64
